In [91]:
import os


In [92]:
print(os.getcwd())

c:\Users\clair\Documents\json-api-geocodage\notebooks


In [93]:
print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\clair\Documents\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json', 'villes.json']


Challenge 1 — Un JSON à deux étages

Etape 1.

In [94]:
import json

with open("../data/villes.json", encoding="utf-8") as f:
    data = json.load(f)

In [95]:
print("Nombres de villes: ",len(data["villes"] ))
print("Deuxième ville:",data["villes"][1]["nom"])
print("Nombre de restaurants à Syracuse:",len(data["villes"][2]["restaurants"]))
print("premier restaurant de Palerme :", data["villes"][1]["restaurants"][0]["nom"])

Nombres de villes:  3
Deuxième ville: Palerme
Nombre de restaurants à Syracuse: 2
premier restaurant de Palerme : Antica Focacceria


Etape 2.

In [96]:
for ville in data["villes"]:
    print(ville["nom"], "possède", len(ville["restaurants"]), "restaurants")

Catane possède 3 restaurants
Palerme possède 2 restaurants
Syracuse possède 2 restaurants


Etape 3.

In [97]:
import pandas as pd

df = pd.json_normalize(data,record_path=["villes","restaurants"],meta=[["villes", "nom"],["villes","region"],["villes","habitants"]])

In [98]:
df

,nom,type,notes,villes.nom,villes.region,villes.habitants
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605


In [99]:
df["note_moyenne"] = df["notes"].apply(lambda x:round(sum(x)/len(x),2))

In [100]:
df

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762,4.70
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762,4.45
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762,4.15
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828,4.27
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828,4.55
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605,4.20


3. Une API, c'est un JSON au bout d'une URL

Etape 3.1

In [101]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [102]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


Etape 3.3

In [103]:
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
print(r.url)

https://api-adresse.data.gouv.fr/search/?q=20+avenue+de+S%C3%A9gur+75007+Paris&limit=1


Etape 3.5

In [104]:
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


Etape 3.6

In [105]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [106]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


Challenge 2 — Interroger une API en série

Etape 1.

In [107]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

In [108]:
url = "https://api-adresse.data.gouv.fr/search/"
r =requests.get(url, params={"q": adresses[0], "limit": 1},timeout=10)  
print(r.status_code)
resultat = r.json()["features"][0]
print(resultat["properties"]["label"])
print(resultat["properties"]["score"])

200
20 Avenue de Ségur 75007 Paris
0.9716281818181818


Etape 2.

In [109]:
resultats = []
for adresse in adresses:
    r = requests.get(url,params={"q": adresse,"limit": 1}, timeout=10)
    features = r.json()["features"]
    if features :
        pros = features[0]["properties"]
        resultats.append({
            "adresse": adresse,
            "label": pros["label"],
            "score": pros["score"],
            "postcode": pros["postcode"],
            "city" : pros["city"] 
        })
    else: 
        resultats.append({
            "adresse_demandee":adresse,
            "label": None,
            "score": None,
            "postcode": None,
            "city": None
        })

In [110]:
resultats

[{'adresse': '20 avenue de Ségur 75007 Paris',
  'label': '20 Avenue de Ségur 75007 Paris',
  'score': 0.9716281818181818,
  'postcode': '75007',
  'city': 'Paris'},
 {'adresse': 'Grand Place, Lille',
  'label': 'Place de Gand 59800 Lille',
  'score': 0.5682435064935065,
  'postcode': '59800',
  'city': 'Lille'},
 {'adresse': '1 rue de la Paix, Paris',
  'label': '1 Rue de la Paix 75002 Paris',
  'score': 0.9640599999999998,
  'postcode': '75002',
  'city': 'Paris'},
 {'adresse': 'place Bellecour, Lyon',
  'label': 'Place Bellecour 69002 Lyon',
  'score': 0.9673672727272726,
  'postcode': '69002',
  'city': 'Lyon'},
 {'adresse_demandee': 'zzzzz qwerty 99999',
  'label': None,
  'score': None,
  'postcode': None,
  'city': None}]

Etape 3.

In [111]:
import pandas as pd

df_adresses = pd.DataFrame(resultats)
df_adresses

,adresse,label,score,postcode,city,adresse_demandee
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris,NaN
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille,NaN
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris,NaN
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon,NaN
4,NaN,NaN,NaN,NaN,NaN,zzzzz qwerty 99999


Etape 4.

In [112]:
print("Nombre d'adresses trouvées:", df_adresses["label"].notna().sum())

Nombre d'adresses trouvées: 4


combien d'adresses ont été trouvées ?
Nous avons trouvées 4 adresses.

In [113]:
print("Score le plus bas:", df_adresses.loc[df_adresses["score"].idxmin()])

Score le plus bas: adresse                    Grand Place, Lille
label               Place de Gand 59800 Lille
score                                0.568244
postcode                                59800
city                                    Lille
adresse_demandee                          NaN
Name: 1, dtype: object


laquelle a le score le plus bas parmi celles trouvées, et pourquoi à ton avis ?
Parce que la recherhce manque de précision, il faudrait une adresse complète .

In [114]:
df_adresses[["adresse","label"]]

,adresse,label
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon
4,NaN,NaN


Les trois questions de l'étape 4 ont une réponse écrite ?
Non ,l'api renvoie une adresse proche 

4.Géocoder : d'une adresse à un point

Etape 4.1

In [115]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
reponse = r.json()

import json
print(json.dumps(reponse, ensure_ascii=False, indent=2)[:600])

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.308628,
          48.850699
        ]
      },
      "properties": {
        "label": "20 Avenue de Ségur 75007 Paris",
        "score": 0.9716281818181818,
        "housenumber": "20",
        "id": "75107_8909_00020",
        "banId": "c575af14-f697-464c-b70b-914cedb239a4",
        "name": "20 Avenue de Ségur",
        "postcode": "75007",
        "citycode": "75107",
        "x": 649266.35,
        "y": 6861406.23,
        "city": 


In [116]:
f = reponse["features"][0]
lon, lat =f["geometry"]["coordinates"]
print("longitude:", lon, "lattitude:", lat)
print("label    :", f["properties"]["label"])

longitude: 2.308628 lattitude: 48.850699
label    : 20 Avenue de Ségur 75007 Paris


Etape 4.3

In [117]:
import pandas as pd

df = pd.json_normalize(reponse, record_path="features")
print(df.columns.tolist())

['type', 'geometry.type', 'geometry.coordinates', 'properties.label', 'properties.score', 'properties.housenumber', 'properties.id', 'properties.banId', 'properties.name', 'properties.postcode', 'properties.citycode', 'properties.x', 'properties.y', 'properties.city', 'properties.district', 'properties.context', 'properties.type', 'properties.importance', 'properties.depcode', 'properties.street', 'properties._type']


In [118]:
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
print(df[["properties.label", "lat", "lon"]])

                 properties.label        lat       lon
0  20 Avenue de Ségur 75007 Paris  48.850699  2.308628


Etape 4.4

In [119]:
def geocoder_fr(adresse):
    """Renvoie (latitude, longitude) pour une adresse française, ou (None, None)."""
    url = "https://api-adresse.data.gouv.fr/search/"
    try:
        r = requests.get(url, params={"q": adresse, "limit": 1}, timeout=10)
        r.raise_for_status()
        features = r.json()["features"]
        if not features:
            return None, None
        lon, lat = features[0]["geometry"]["coordinates"]
        return lat, lon
    except requests.exceptions.RequestException:
        return None, None

In [120]:
print(geocoder_fr("20 avenue de Ségur 75007 Paris"))
print(geocoder_fr("zzzzz qwerty 99999"))

(48.850699, 2.308628)
(None, None)


5. Nominatim : le monde entier, une requête par seconde

Etape 5.1

In [121]:
url = "https://nominatim.openstreetmap.org/search"
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}

r = requests.get(url, params=params, headers=headers, timeout=10)
resultats = r.json()

print(resultats[0]["lat"], resultats[0]["lon"])
print(resultats[0]["display_name"])

37.5111425 15.0950365
Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia


In [122]:
import pandas as pd
df = pd.json_normalize(resultats)
print(df[["lat", "lon", "display_name"]])

          lat         lon                                       display_name
0  37.5111425  15.0950365  Via Pietro Mascagni, Picanello-Ognina-Barriera...


Etape 5.2

In [123]:
headers = {
    "User-Agent": "bootcamp-simplon-da/1.0 (throuduskarlus@gmail.com)"
}

Etape 5.3

In [124]:
import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (throuduskarlus@gmail.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [125]:
import time
debut = time.time()
print(geocoder_monde("54 Via Pietro Mascagni, Catania, Italy"))
print(geocoder_monde("Piazza del Duomo, Catania, Italy"))
print(f"durée totale : {time.time() - debut:.1f} s")

(37.5111425, 15.0950365)
(37.5024076, 15.0871898)
durée totale : 2.5 s


6. Afficher les points sur une carte

In [126]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [127]:
import folium

point = [37.5112, 15.0949]          # latitude, longitude — Catane
m = folium.Map(location=point, zoom_start=14)
folium.Marker(location=point, popup="Un bon restaurant").add_to(m)
m

Challenge 3 — les restaurants de Catane, sur une carte


Etape 1 et 2.

In [128]:
import json
import pandas as pd

with open("../data/restaurants.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
df = pd.DataFrame(data["restaurants"])
df["ville"] = data["ville"]
df["pays"] = data["pays"]
df["note_moyenne"] = df["notes"].apply(lambda notes: round(sum(notes)/len(notes),2))
df = df[["nom", "adresse", "type", "ville", "pays", "note_moyenne"]]
df

,nom,adresse,type,ville,pays,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,4.00


Etape 3.

In [129]:
import time
import requests

def geocoder(adresse):
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (throuduskarlus@gmail.com)"}
    params = {"q": "adresse", "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        resultat = r.json()
        if not resultat:
            return (None, None)
        return float(resultat[0]["lat"]),float(resultat[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

latitudes = []
longitudes = []

for index, ligne in df.iterrows():
    
    adresse_complete = f"{ligne['adresse']}, {ligne['ville']}, {ligne['pays']}"
    lat, lon = geocoder(adresse_complete)
    latitudes.append(lat)
    longitudes.append(lon)


df["lat"] = latitudes
df["lon"] = longitudes

df[["nom", "adresse", "lat", "lon"]]


,nom,adresse,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",49.505248,0.080565
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",49.505248,0.080565
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",49.505248,0.080565
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",49.505248,0.080565


Étape 4 — la carte

In [130]:
import folium
df["lat"], df["lon"] = df["lon"], df["lat"]        
m = folium.Map(location = [df.loc[0,"lat"], df.loc[0,"lon"]], zoom_start = 15)
for _, ligne in df.iterrows():
    if pd.notna(ligne["lat"]) and pd.notna(ligne["lon"]):
        infobulle = f"{ligne['nom']} — Note : {ligne['note_moyenne']}/5"
        
        folium.Marker(
            location=[ligne["lat"], ligne["lon"]],
            tooltip=infobulle,
            popup=infobulle
        ).add_to(m)
m